# FastAPI 기반 TTS 실습

**목표:** FastAPI REST API를 만들고 텍스트를 MP3로 변환하여 HTML 화면에서 재생한다.

프로젝트의 `main.py`, `templates/index.html`, `static/app.js`, `static/style.css`를 함께 사용한다. API 호출은 OpenAI 키와 네트워크 연결이 필요하다.

## 1. 라이브러리 설치

현재 Jupyter 커널의 Python 환경에 설치한다.

In [ ]:
%pip install -U fastapi "uvicorn[standard]" openai python-dotenv jinja2 httpx ipython

## 2. 프로젝트 파일 확인

In [ ]:
from pathlib import Path
base_dir = Path.cwd()
required = ["main.py", "templates/index.html", "static/app.js", "static/style.css"]
for name in required:
    print(name, "OK" if (base_dir / name).exists() else "없음")

## 3. API 키 설정

`.env.example`을 `.env`로 복사하고 실제 API 키를 입력한다. 노트북 출력에 키를 표시하지 않는다.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
env_file = Path(".env")
if not env_file.exists():
    env_file.write_text("OPENAI_API_KEY=your_api_key_here\n", encoding="utf-8")
    print(".env 파일을 생성했습니다. 실제 API 키를 입력하고 이 셀을 다시 실행하세요.")
load_dotenv(env_file, override=True)
key = os.getenv("OPENAI_API_KEY", "")
print("API 키 설정:", "완료" if key and key != "your_api_key_here" else "미설정")

## 4. FastAPI 앱 불러오기

`main.py`에는 `GET /`와 `POST /api/tts`가 구현되어 있다. 실제 TTS 호출 전 입력 검증을 먼저 확인한다.

In [1]:
from main import app, TTSRequest
print("앱 제목:", app.title)
print("등록된 경로:", [r.path for r in app.routes])
request_data = TTSRequest(text="안녕하세요. TTS 실습입니다.", voice="alloy", speed=1.0)
print(request_data.model_dump())

앱 제목: FastAPI TTS
등록된 경로: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/static', '/', '/api/tts']
{'text': '안녕하세요. TTS 실습입니다.', 'voice': 'alloy', 'speed': 1.0}


## 5. API 테스트: 입력 검증

이 테스트는 OpenAI API를 호출하지 않는다.

In [2]:
from fastapi.testclient import TestClient
client = TestClient(app)
response = client.post("/api/tts", json={"text": "  ", "voice": "alloy", "speed": 1.0})
print("HTTP 상태 코드:", response.status_code)
print("응답:", response.json())
assert response.status_code == 422

HTTP 상태 코드: 422
응답: {'detail': '변환할 텍스트를 입력하세요.'}


## 6. 음성 생성 API 테스트

**주의:** 이 셀은 실제 OpenAI API를 호출하며 사용량에 따른 비용이 발생할 수 있다. 키가 없으면 실행하지 않는다.

In [3]:
from IPython.display import Audio, display
from pathlib import Path
import os
key = os.getenv("OPENAI_API_KEY", "")
if not key or key == "your_api_key_here":
    print("API 키를 먼저 설정하세요.")
else:
    response = client.post("/api/tts", json={
        "text": "안녕하세요. FastAPI 기반 텍스트 음성 변환 실습입니다.",
        "voice": "alloy",
        "speed": 1.0
    })
    print("HTTP 상태 코드:", response.status_code)
    if response.status_code == 200:
        output_dir = Path("outputs")
        output_dir.mkdir(exist_ok=True)
        output_file = output_dir / "notebook_tts.mp3"
        output_file.write_bytes(response.content)
        print("파일 저장:", output_file.resolve())
        display(Audio(filename=str(output_file)))
    else:
        print("오류:", response.text[:1000])

HTTP 상태 코드: 200
파일 저장: C:\sk-encoa\llm-workspace\fastapi_tts_project\outputs\notebook_tts.mp3


## 7. FastAPI 서버 실행

노트북에서 서버를 실행하려면 아래 셀을 사용한다. 이미 8000번 포트를 사용 중이면 실행되지 않는다. 서버를 중지하려면 마지막 셀을 실행한다.

In [6]:
import subprocess
import sys
from pathlib import Path
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(Path.cwd()),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)
print("서버 프로세스 PID:", server_process.pid)
print("웹 화면: http://127.0.0.1:8000")
print("API 문서: http://127.0.0.1:8000/docs")

서버 프로세스 PID: 12828
웹 화면: http://127.0.0.1:8000
API 문서: http://127.0.0.1:8000/docs


## 8. 서버 상태 확인

In [7]:
import httpx
try:
    r = httpx.get("http://127.0.0.1:8000/", timeout=10)
    print("HTTP 상태 코드:", r.status_code)
except Exception as e:
    print("서버 연결 실패:", e)

HTTP 상태 코드: 200


## 9. 서버 종료

노트북에서 실행한 서버만 종료한다.

In [8]:
if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()
    print("서버 종료 완료")
else:
    print("실행 중인 노트북 서버가 없습니다.")

실행 중인 노트북 서버가 없습니다.


## 정리

- `POST /api/tts`는 JSON 요청을 받아 MP3 바이트를 반환한다.
- `StreamingResponse(BytesIO(...))`는 생성된 음성 전체를 메모리에서 반환한다.
- 실제 공개 서비스에는 인증, 사용량 제한, 비용 관리가 추가로 필요하다.